<div style="background-color:#007033; color:white; padding:18px 22px; border-radius:6px; margin-bottom:10px;"><h1 style="color:white; margin:0 0 4px 0;">Week 3 — Cloud Foundations and Reference Architecture</h1><div style="font-size:15px; opacity:0.9;">FraudGuard Case Study · Stage 3 of 10 · INFT 41000, Omar Altrad, Durham College</div></div>

<div style="background-color:#E0F1E8; border-left:5px solid #007033; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#007033;">This week</b><br>Week 2 picked a deployment environment on paper. This week makes it real, in two parts: first, create an AWS account the right way (not with root credentials), connect Python to it with <code>boto3</code>, and put FraudGuard's dataset in real cloud storage — the foundation every later AWS week builds on. Second, step back and draw the whole picture: how that AWS setup, plus the pieces every later week will build, fit together into one reference architecture, and where the risky single points of failure are.</div>

| # | Section |
|---|---|
| 0 | Setup |
| 1 | Load Week 2's recommendation |
| 2 | AWS account & IAM setup (one-time) |
| 3 | Verify your connection |
| 4 | The multi-cloud landscape |
| 5 | Storage: put FraudGuard's data in S3 |
| 6 | IAM: check what your user can actually do |
| 7 | Compute & managed AI services (preview) |
| 8 | The Cloud Adoption Framework |
| 9 | What is a reference architecture? |
| 10 | Define FraudGuard's components |
| 11 | Draw the architecture diagram |
| 12 | Find the integration points |
| 13 | Single points of failure |
| 14 | Save your work |

<div style="background-color:#FCF3D9; border-left:5px solid #B58800; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B58800;">Setup</b><br>Same shared project as Week 1–2, plus a new package this week (<code>boto3</code>) — already added to <code>requirements.txt</code>; run <code>pip install -r requirements.txt</code> again before starting. First time in this project? Run <code>00_Setup.ipynb</code> first.</div>

In [ ]:
import sys
from pathlib import Path

def find_project_root():
    '''Walk up from the current working directory until we find the
    Course_Project root (identified by having both requirements.txt
    and a src/ folder). Works no matter which VS Code cwd setting is in
    effect for notebooks (fileDirname or workspaceFolder), because either
    way this notebook lives somewhere inside the project tree.'''
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the Course_Project root (a folder containing both "
        "requirements.txt and src/). Open this notebook from inside the project "
        "folder in VS Code (File > Open Folder > Course_Project), or set PROJECT_ROOT "
        "manually, e.g. PROJECT_ROOT = Path(r'C:\path\to\INFT41000\Course_Project')."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"

for required_dir in (DATA_DIR, MODELS_DIR):
    if not required_dir.is_dir():
        raise FileNotFoundError(
            f"Missing folder: {required_dir}\n"
            "Create it first (see the folder-structure instructions in 00_Setup.ipynb) -- "
            "this notebook does not create folders for you."
        )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import json
import filecmp
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

print("Setup OK")

## 1. Load Week 2's Recommendation

In [ ]:
REC_PATH = DATA_DIR / "environment_recommendation.txt"

if not REC_PATH.exists():
    raise FileNotFoundError(
        f"Missing {REC_PATH.name} in {REC_PATH.parent}. Run Week 2's notebook first "
        "(the \"Save Your Work\" section) -- this notebook does not regenerate it for you."
    )

print(REC_PATH.read_text())

<div style="background-color:#DBF2FF; border-left:5px solid #005291; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#005291;">Whatever you recommended, this week still applies</b><br>Even an On-Prem or Hybrid recommendation needs cloud skills — almost every real deployment includes a cloud component somewhere (backups, burst capacity, managed services). We build hands-on AWS skills either way, starting now.</div>

## 2. AWS Account & IAM Setup (One-Time)

<div style="background-color:#FDE8E6; border-left:5px solid #B22222; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B22222;">Real account, real safety rules — read before continuing</b><br>This week uses a <b>real AWS account</b> (Free Tier). Follow these rules the whole course:<br>1. <b>Never use your AWS root account's access keys for anything.</b> Root is for account administration only.<br>2. <b>Never paste an access key or secret key into a notebook cell.</b> Configure them once via the AWS CLI (Step 2c below) — <code>boto3</code> finds them automatically.<br>3. <b>Set a budget alert</b> (Step 2d) so you get emailed before you're ever charged.<br>4. Everything we do this week (a few KB of S3 storage) stays inside the Free Tier's 5GB — this is not where cloud bills come from. The real risk is a forgotten EC2/SageMaker instance left <i>running</i> — we don't create any this week, and Week 5 will cover shutting those down when we do.</div>

**a. Create an AWS Free Tier account** (skip if you already have one): go to
[aws.amazon.com/free](https://aws.amazon.com/free), sign up (requires a credit card, but
Free Tier–eligible usage isn't charged). Enable MFA on the root user right after signup
(Account name, top-right → Security credentials → Assign MFA device).

**b. Create an IAM user for this course** (in the AWS Console → IAM → Users → Create user):
- User name: `inft41000-student` (or your own choice)
- Attach policy directly: `AmazonS3FullAccess` and `IAMReadOnlyAccess` (enough for this
  week; we'll add more as later weeks need it)
- Create the user, then open it → **Security credentials** tab → **Create access key** →
  choose "Command Line Interface (CLI)" → download the `.csv`. This is the only time the
  secret key is shown — keep the file, don't share it, don't commit it anywhere.

**c. Install and configure the AWS CLI:**
```powershell
# Windows
winget install Amazon.AWSCLI
```
```bash
# macOS
brew install awscli
```
Then, from a terminal, run `aws configure` and paste in the Access Key ID / Secret Access
Key from the `.csv`, plus a default region (use `us-east-1` to match this notebook) and
output format `json`. This writes to `~/.aws/credentials` — `boto3` reads it automatically,
which is why no key ever appears in code below.

**d. Set a budget alert** (Console → Billing → Budgets → Create budget → Zero spend budget,
or a $5 threshold) so you get an email if anything unexpected starts costing money.

## 3. Verify Your Connection

`boto3` reads the credentials `aws configure` saved — nothing to paste here. This confirms
you're connected as your **IAM user**, not root (check the ARN below: it should say
`user/inft41000-student`, never `root`).

In [ ]:
import boto3

REGION = "us-east-1"
sts = boto3.client("sts", region_name=REGION)
identity = sts.get_caller_identity()

print(f"Account: {identity['Account']}")
print(f"ARN:     {identity['Arn']}")

assert ":user/" in identity["Arn"], (
    "This looks like the root account, not an IAM user -- go back to Step 2b and use an "
    "IAM user's access keys instead."
)
print("\nConnected as an IAM user -- good to go.")

## 4. The Multi-Cloud Landscape

This course goes hands-on with AWS (matching Week 2's real cost/decision-matrix exercise),
but the same concepts exist on every major cloud under different names:

| Capability | AWS | Azure | Google Cloud | Service model |
|---|---|---|---|---|
| Compute (VMs) | EC2 | Virtual Machines | Compute Engine | IaaS |
| Storage | S3 | Blob Storage | Cloud Storage | IaaS |
| Networking | VPC | Virtual Network (VNet) | VPC | IaaS |
| GPUs | EC2 P/G-series | NC/ND-series VMs | Compute Engine + A100/T4 GPUs | IaaS |
| Managed AI/ML | SageMaker | Azure Machine Learning | Vertex AI | PaaS |
| Containers | ECS / EKS (+ Fargate) | Container Apps / AKS | Cloud Run / GKE | PaaS |
| Serverless functions | Lambda | Functions | Cloud Functions | PaaS (serverless) |

FraudGuard's model is a tabular RandomForest — it doesn't need a GPU. GPUs matter far more for
deep learning/LLM workloads, but knowing where they'd come from on each cloud is worth having.

**Service model** is the other axis worth tracking here: **IaaS** (Infrastructure as a Service) hands you
raw compute/storage/network and you manage everything above that; **PaaS** (Platform as a Service) also
manages the runtime, scaling, and patching for you, at the cost of some control. (**SaaS** — a finished
application you just use, like Gmail — doesn't appear in this table because nothing here is a finished
application; it shows up again in Section 7.)

## 5. Storage: Put FraudGuard's Data in S3

### Exercise 3.1
Using your AWS account ID (from Section 3) as part of the bucket name guarantees a
**globally unique** name (S3 bucket names must be unique across *all* AWS accounts, not just
yours). Create the bucket (handle the case where it already exists, so re-running this cell
is safe), upload Week 1's dataset, list what's in the bucket, then download it back and
verify the downloaded file is identical to the original.

In [ ]:
# YOUR CODE HERE
BUCKET_NAME = f"fraudguard-{identity['Account']}"
S3_KEY = "raw/fraudguard_transactions_week1.csv"
LOCAL_DATASET = DATA_DIR / "fraudguard_transactions_week1.csv"

s3 = boto3.client("s3", region_name=REGION)

# 1. Create the bucket (handle BucketAlreadyOwnedByYou for safe re-runs)
raise NotImplementedError

# 2. Upload LOCAL_DATASET to S3_KEY

# 3. List the bucket's contents and print each object's key/size/last-modified

# 4. Download it back to DOWNLOAD_PATH and assert it matches LOCAL_DATASET with filecmp.cmp

<div style="background-color:#DBF2FF; border-left:5px solid #005291; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#005291;">Storage classes (for later)</b><br>Everything above used S3 Standard. Once data is old enough that fast access rarely matters, Standard-IA or Glacier cost far less per GB — worth knowing for Week 8 (cost optimization), not needed yet.</div>

## 6. IAM: Check What Your User Can Actually Do

Least-privilege is a real production security practice (Week 6 goes deeper): your IAM user
should only have the permissions it needs, and it's good habit to actually check.

### Exercise 3.2
List the policies attached to your IAM user (parse the username from your ARN — everything
after the last `/`) and print each policy's name and ARN.

In [ ]:
# YOUR CODE HERE
iam = boto3.client("iam", region_name=REGION)
username = identity["Arn"].split("/")[-1]

raise NotImplementedError

*Both `AmazonS3FullAccess` and `IAMReadOnlyAccess` are broader than a real production system should use long-term. What's the narrowest S3 permission set FraudGuard's IAM user would actually need, based only on what Exercise 3.1 did?*

## 7. Compute & Managed AI Services (Preview — Not Built Yet)

We are **not** launching any compute this week (EC2/SageMaker instances cost money while
they run, and there's nothing to deploy onto them until Week 5 packages FraudGuard as a
service). For now, know the shape of the decision:

| Option | Best for | Trade-off | Service model |
|---|---|---|---|
| EC2 | Full control over the server | You patch, secure, and manage it yourself | IaaS |
| SageMaker | Managed training + real-time inference endpoints | Less control, more built-in ML tooling (Week 9) | PaaS |
| Lambda (serverless) | Small, infrequent, fast requests | Cold starts, execution time/size limits | PaaS (serverless) |

FraudGuard's model is small and needs low-latency scoring — Week 5 revisits this table with
real numbers once there's an actual service to deploy.

None of these three is **SaaS** — SaaS is a finished application someone else runs entirely (Gmail,
Salesforce): there's no "deploy your model to it" step because you're not deploying anything, you're
just using it. FraudGuard is being *built*, not bought, so IaaS and PaaS are the only options on the table.

## 8. The Cloud Adoption Framework: Where FraudGuard Is Headed

Choosing IaaS vs. PaaS vs. SaaS (Sections 4 and 7) is one decision inside a bigger one: **how
does an organization move onto the cloud at all, in a way that doesn't turn into chaos?** Cloud
providers publish **Cloud Adoption Frameworks** (AWS CAF, Microsoft's Cloud Adoption Framework
for Azure, Google Cloud's Adoption Framework) — structured phases that answer exactly that
question. They differ in the details, but the shape is the same everywhere:

| Phase | Question it answers | Where FraudGuard already is |
|---|---|---|
| **Plan** | What are we moving, and why? What's the business case? | Week 2's environment decision — the cost/decision matrix that chose this environment in the first place |
| **Ready** | Do we have the accounts, permissions, and guardrails to do this safely? | This week's Sections 2–3 — the AWS account, the IAM user, the budget alert |
| **Adopt** (migrate/build) | Build or move the actual workload | This week's S3 bucket onward — Weeks 4–9 build the rest |
| **Govern & Operate** | Once it's live, who's watching it, and who can touch what? | Section 6's IAM permissions check now; Week 9's monitoring later |

FraudGuard is one small project, so it moves through these phases in weeks instead of the
months or years a real enterprise migration takes — but the phases themselves don't change with
scale. A framework like this is also *why* the reference architecture in Section 9 exists:
"Adopt" needs a map of what's being built, and "Govern & Operate" needs to know what to watch.


## 9. What Is a Reference Architecture?

Now that real AWS infrastructure exists, it's worth stepping back before building anything
else: a **reference architecture** is a diagram (plus a short description of each piece)
showing every major component of a system, grouped by layer, with arrows showing which
components talk to which. It answers three questions at a glance that no amount of code can:

- **What exists?** Every component the system depends on, named once.
- **Who talks to whom?** The arrows — this is where hidden dependencies get caught *before*
  they cause an incident.
- **What's the blast radius if one piece goes down?** A component with many incoming arrows
  and no backup is a single point of failure (Section 13).

FraudGuard's architecture groups components into six layers:

| Layer | Meaning | FraudGuard example so far |
|---|---|---|
| External | Systems we don't own or control | The bank's core payments system |
| Application/API | Request handling, auth, routing | The scoring API (Week 5) |
| Model | The trained model and its versions | The fraud model (Week 1), model registry (Week 9) |
| Data | Storage and data processing | S3 bucket (this week), feature pipeline (Week 4) |
| Infrastructure | Compute and networking that hosts everything else | Compute cluster, load balancer (Weeks 5, 8) |
| Monitoring | Watching the system once it's live | Metrics, drift detection (Week 9) |

Every component below is tagged with **which week of this course actually builds it** — this
diagram is FraudGuard's finish line, drawn now so every later week has a map to build toward.

## 10. Define FraudGuard's Components

### Exercise 3.3
Below is a partial component list. Each component is a dictionary with `name`, `layer`,
`technology`, `responsibility`, `connects_to` (a list of component names this one sends
data/requests **to** — the arrow points from this component toward each name in the list),
and `built_in_week` (which week of this course builds it; use `"N/A (external)"` for systems
outside our control). Complete the list using the spec table that follows.

| name | layer | technology | connects_to | built_in_week |
|---|---|---|---|---|
| Core Payments System | External | Existing bank/merchant systems | Auth Gateway | N/A (external) |
| Auth Gateway | Application/API | API Gateway + IAM (AWS) | Scoring API | 6 |
| Scoring API | Application/API | FastAPI | Fraud Model, Metrics & Logs | 5 |
| Fraud Model | Model | scikit-learn RandomForest | Model Registry | 1 |
| Model Registry | Model | MLflow Model Registry | *(none)* | 9 |
| S3 Raw Bucket | Data | AWS S3 | Feature Pipeline | 3 |
| Feature Pipeline | Data | pandas batch job | Model Registry, Drift Monitor | 4 |
| Compute Cluster | Infrastructure | AWS ECS/EKS | Scoring API, Metrics & Logs | 5 |
| Load Balancer | Infrastructure | AWS ELB + Auto Scaling | Compute Cluster | 8 |
| Metrics & Logs | Monitoring | AWS CloudWatch | Drift Monitor | 9 |
| Drift Monitor | Monitoring | Evidently AI | *(none)* | 9 |

The first two rows are filled in below as an example — add the remaining nine.

In [ ]:
# YOUR CODE HERE
# Complete the list using the spec table above. The first two rows are done for you.
ARCHITECTURE = [
    {"name": "Core Payments System", "layer": "External", "technology": "Existing bank/merchant systems",
     "responsibility": "Sends live transactions to be scored", "connects_to": ["Auth Gateway"], "built_in_week": "N/A (external)"},
    {"name": "Auth Gateway", "layer": "Application/API", "technology": "API Gateway + IAM (AWS)",
     "responsibility": "Authenticates callers, routes requests", "connects_to": ["Scoring API"], "built_in_week": 6},
    # Add the remaining 9 components from the spec table above.
]

raise NotImplementedError

LAYER_ORDER = ["External", "Application/API", "Model", "Data", "Infrastructure", "Monitoring"]
print(f"{len(ARCHITECTURE)} components defined across {len(LAYER_ORDER)} layers")

## 11. Draw the Architecture Diagram

The function below (already written — no exercise here, just run it) lays out each layer as
a horizontal row, draws each component as a box colored by layer, and draws an arrow for
every `connects_to` relationship. It reuses the same brand color palette as the slide decks
and Week 2's charts, so colors mean the same thing everywhere in this course.

In [ ]:
LAYER_COLORS = {
    "External": "#5A6066",
    "Application/API": "#005291",
    "Model": "#B58800",
    "Data": "#007033",
    "Infrastructure": "#5A6066",
    "Monitoring": "#B22222",
}


def render_architecture(architecture, layer_order, layer_colors=LAYER_COLORS, save_path=None):
    by_layer = {layer: [c for c in architecture if c["layer"] == layer] for layer in layer_order}
    positions = {}
    row_h = 1.0
    for row_idx, layer in enumerate(layer_order):
        y = -row_idx * row_h
        comps = by_layer[layer]
        col_w = 3.4
        start_x = -(len(comps) - 1) * col_w / 2.0
        for col_idx, comp in enumerate(comps):
            positions[comp["name"]] = (start_x + col_idx * col_w, y)

    fig, ax = plt.subplots(figsize=(13, 8))

    for comp in architecture:
        x, y = positions[comp["name"]]
        color = layer_colors[comp["layer"]]
        box = FancyBboxPatch((x - 1.5, y - 0.28), 3.0, 0.56,
                              boxstyle="round,pad=0.05,rounding_size=0.08",
                              linewidth=2, edgecolor=color, facecolor="white", zorder=3)
        ax.add_patch(box)
        ax.text(x, y, comp["name"], ha="center", va="center", fontsize=10.5,
                 fontweight="bold", color=color, zorder=4)

    for comp in architecture:
        x1, y1 = positions[comp["name"]]
        for target_name in comp["connects_to"]:
            x2, y2 = positions[target_name]
            arrow = FancyArrowPatch((x1, y1), (x2, y2), connectionstyle="arc3,rad=0.15",
                                     arrowstyle="-|>", mutation_scale=16, linewidth=1.4,
                                     color="#8A8F96", zorder=2, shrinkA=22, shrinkB=22)
            ax.add_patch(arrow)

    for row_idx, layer in enumerate(layer_order):
        y = -row_idx * row_h
        ax.text(-7.6, y, layer, ha="left", va="center", fontsize=12,
                 fontweight="bold", color=layer_colors[layer])

    ax.set_xlim(-8.2, 8.2)
    ax.set_ylim(-row_h * (len(layer_order) - 1) - 0.8, 0.8)
    ax.axis("off")
    ax.set_title("FraudGuard Reference Architecture", fontsize=15, fontweight="bold", pad=14)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=130)
    plt.show()


render_architecture(ARCHITECTURE, LAYER_ORDER)

<div style="background-color:#DBF2FF; border-left:5px solid #005291; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#005291;">Reading the diagram</b><br>Follow the arrows top to bottom: a transaction enters at <b>Core Payments System</b>, gets authenticated, scored by the model, and the outcome is logged and watched for drift. Notice <b>Model Registry</b> and <b>Drift Monitor</b> have no outgoing arrows — they're endpoints, not pass-through components.</div>

## 12. Find the Integration Points

An **integration point** is any connection that crosses a layer boundary (for example, Data
→ Model). These are exactly where most real production incidents happen — a format change,
a timeout, or a permissions error at a boundary between two teams' components — so knowing
where they are is worth more than the diagram's visual polish.

### Exercise 3.4
Write `find_integration_points(architecture)` that returns a list of `(source, target)` name
pairs for every `connects_to` edge where the source and target are in **different** layers
(a same-layer edge, like Auth Gateway → Scoring API, is not an integration point).

In [ ]:
# YOUR CODE HERE
def find_integration_points(architecture):
    raise NotImplementedError


integration_points = find_integration_points(ARCHITECTURE)
total_edges = sum(len(c["connects_to"]) for c in ARCHITECTURE)

print(f"{len(integration_points)} of {total_edges} total connections cross a layer boundary:\n")
for source, target in integration_points:
    print(f"  {source}  -->  {target}")

## 13. Single Points of Failure

A **single point of failure (SPOF)** is a component that, if it goes down, takes the whole
system (or a critical path through it) down with it — with no redundancy to fall back on.
Look at the diagram and the connections above, then answer in the markdown cell below:

1. Name the single biggest SPOF risk in FraudGuard's architecture as drawn (which component,
   and why).
2. FraudGuard's model rejects a live transaction if the Scoring API can't reach the Fraud
   Model in time. What should happen instead — fail open (approve the transaction), fail
   closed (decline it), or something else? Justify your answer against Week 2's Risk
   criterion (15% weight in the environment decision matrix).
3. Which future week's build (from the `built_in_week` field) most directly reduces one of
   these risks, and how?

*Your answers here — this is a written reflection, not a code exercise.*

## 14. Save Your Work — Needed for Later Weeks

In [ ]:
aws_config = {
    "account_id": identity["Account"],
    "region": REGION,
    "bucket_name": BUCKET_NAME,
    "dataset_key": S3_KEY,
}
with open(DATA_DIR / "aws_config.json", "w") as f:
    json.dump(aws_config, f, indent=2)

with open(DATA_DIR / "architecture.json", "w") as f:
    json.dump({"layer_order": LAYER_ORDER, "components": ARCHITECTURE}, f, indent=2)

print(f"Saved: {(DATA_DIR / 'aws_config.json').resolve()}")
print(json.dumps(aws_config, indent=2))
print(f"\nSaved: {(DATA_DIR / 'architecture.json').resolve()}")
print(f"{len(ARCHITECTURE)} components, {len(find_integration_points(ARCHITECTURE))} integration points")

<div style="background-color:#E0F1E8; border-left:5px solid #007033; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#007033;">Submit</b><br>This notebook, run top-to-bottom with no errors · Exercises 3.1–3.4 completed · the IAM reflection answered · a screenshot or note confirming your budget alert (Step 2d) is set · the rendered diagram · the three-part SPOF reflection (Section 13).<br><br><b>Next week:</b> Week 4 — building the real Feature Pipeline component from this diagram, turning raw S3 data into model-ready features.</div>